In [1]:
import os
from dotenv import load_dotenv

load_dotenv('../.env')

required = [
    "LANGCHAIN_TRACING_V2",
    "LANGCHAIN_ENDPOINT",
    "LANGCHAIN_API_KEY",
    "LANGCHAIN_PROJECT"
]

print("Langsmith environment check:")
for var in required:
    val = os.getenv(var)
    if val:
        #Mask API key
        display = val[:8] + "..." if "KEY" in var else val
        print(f" {var} = {display}")
    else:
        print(f" {var} = NOT SET")

print()
print("If all four show ✓ — LangSmith tracing is ready.")
print("Every LangChain call you make will now be automatically traced.")


Langsmith environment check:
 LANGCHAIN_TRACING_V2 = true
 LANGCHAIN_ENDPOINT = https://api.smith.langchain.com
 LANGCHAIN_API_KEY = lsv2_pt_...
 LANGCHAIN_PROJECT = docmind

If all four show ✓ — LangSmith tracing is ready.
Every LangChain call you make will now be automatically traced.


In [3]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chat = ChatOllama(model="llama3.2")
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You ar a concise assistant. Answer in one sentence"),
    ("human", "{question}")
])

chain = prompt | chat | parser

response = chain.invoke(
    {"question": " What is a vector DB"},
    config={"run_name": "first-traced-call"}
)

print (f"Response: {response}")
print ()
print("Now go to smith.langchain.com")
print("Open the 'docmind' project")
print("You should see a trace appear within a few seconds")

Response: A Vector Database (VDB) is a type of database that uses dense vector space models to store and query large amounts of high-dimensional data, typically for tasks like faceted search, recommendation systems, and text analysis.

Now go to smith.langchain.com
Open the 'docmind' project
You should see a trace appear within a few seconds


In [5]:
import os
from dotenv import load_dotenv
load_dotenv('../app.py')

import sys
sys.path.append('..')
from src.rag import RAGPipeline
from langsmith import traceable

rag = RAGPipeline()
rag.load_pdf("../data/Fundamentals of Data Engineering.pdf")
print(f"Loaded - {len(rag.chunks)} chunks\n")

@traceable(name="faiss-retrieval")
def traced_retrieval(question: str, k: int =3) -> dict:
    """Retrieval step - traced separately to see latency"""
    import numpy as np
    query_emb = rag.embedding_model.encode([question]).astype(np.float32)
    distances, indices = rag.index.search(query_emb, k)
    chunks = [rag.chunks[i] for i in indices[0]]
    return {
        "question":  question,
        "chunks":    chunks,
        "distances": distances[0].tolist()
    }

@traceable(name="llm-generation")
def traced_generation(question: str, context: str) -> str:
    """Generation step - traced separately to see prompt and response"""
    from langchain_ollama import ChatOllama
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser

    llm = ChatOllama(model="llama3.2")
    parser = StrOutputParser()
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a helpful assistant that answers questions 
based strictly on the provided context. If the answer is not in the 
context, say 'I don't find this in the document'.

Context:
{context}"""),
        ("human", "{question}")
    ])

    chain = prompt | llm | parser
    return chain.invoke({"context" : context, "question" : question})

@traceable(name="docmind-full-query")
def traced_rag_query(question : str) -> str:
    """Full RAG pipeline - parent trace containing retrieval and generation"""
    # Step 1 - Retrieve
    retrieval = traced_retrieval(question)

    # Step 2 - Generate
    context = "\n\n---\n\n".join(retrieval["chunks"])
    answer = traced_generation(question, context)

    return {
        "question": question,
        "answer":   answer,
        "sources":  retrieval["chunks"],
    }

# Run three queries
questions = [
    "What is data engineering?",
    "What are the principles of good data architecture?",
    "Explain the difference between monolith and modular systems.",
]

for question in questions:
    print(f"Q: {question}")
    result = traced_rag_query(question)
    print(f"A: {result['answer'][:150]}...")
    print()

print("Check smith.langchain.com — you should see 3 new traces")
print("Click into one and expand the child spans to see retrieval vs generation latency")

python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 2
python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 10
python-dotenv could not parse statement starting at line 11
python-dotenv could not parse statement starting at line 14
python-dotenv could not parse statement starting at line 16
python-dotenv could not parse statement starting at line 19
python-dotenv could not parse statement starting at line 26
python-dotenv could not parse statement starting at line 28
python-dotenv could not parse statement starting at line 30
python-dotenv could not parse statement starting at line 31
python-dotenv could not parse statement startin

Loaded - 1416 chunks

Q: What is data engineering?
A: Data engineering is a set of operations aimed at creating interfaces and mechanisms for the flow and access of information, which takes dedicated spec...

Q: What are the principles of good data architecture?
A: The principles of good data architecture are not explicitly listed in the provided context. However, the text does mention two key aspects:

1. Choosi...

Q: Explain the difference between monolith and modular systems.
A: According to the context, a monolith system is self-contained, often performing multiple functions under a single system. This means that everything i...

Check smith.langchain.com — you should see 3 new traces
Click into one and expand the child spans to see retrieval vs generation latency


In [6]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')

import sys
sys.path.append('..')
from langsmith import traceable
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
import faiss
import numpy as np

# ── Load PDF once ──────────────────────────────────────────────────────────
reader    = PdfReader("../data/Fundamentals of Data Engineering.pdf")
full_text = ""
for page in reader.pages:
    full_text += page.extract_text() or ""

def build_index(chunks, model_name="all-MiniLM-L6-v2"):
    model      = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, show_progress_bar=False)
    index      = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings.astype(np.float32))
    return model, index

def retrieve(question, model, index, chunks, k=3):
    query_emb          = model.encode([question]).astype(np.float32)
    distances, indices = index.search(query_emb, k)
    return [chunks[i] for i in indices[0]], distances[0].tolist()

def generate(question, context, system_prompt):
    llm    = ChatOllama(model="llama3.2")
    parser = StrOutputParser()
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human",  "{question}")
    ])
    return (prompt | llm | parser).invoke({
        "context":  context,
        "question": question
    })

# ── FAILURE 1: Chunk size too small ───────────────────────────────────────
@traceable(name="failure-1-tiny-chunks")
def failure_tiny_chunks(question: str) -> dict:
    """
    Bug: chunk_size=50 — chunks are so small they lose all context.
    Each chunk is just a sentence fragment.
    Expected failure: retrieved chunks won't contain enough info to answer.
    """
    splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)
    chunks   = splitter.split_text(full_text)
    model, index = build_index(chunks)
    retrieved, distances = retrieve(question, model, index, chunks)
    context  = "\n\n---\n\n".join(retrieved)
    answer   = generate(question, context,
                        "Answer based on context:\n{context}")
    return {"answer": answer, "chunks": retrieved, "n_chunks": len(chunks)}

# ── FAILURE 2: Vague system prompt ────────────────────────────────────────
@traceable(name="failure-2-vague-prompt")
def failure_vague_prompt(question: str) -> dict:
    """
    Bug: system prompt has no context injection and no grounding instruction.
    Expected failure: LLM answers from general knowledge, ignores document.
    """
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks   = splitter.split_text(full_text)
    model, index = build_index(chunks)
    retrieved, distances = retrieve(question, model, index, chunks)
    context  = "\n\n---\n\n".join(retrieved)

    # Bug: context is never injected into the prompt
    answer = generate(question, context,
                      "You are a helpful assistant. Answer the question.")
    return {"answer": answer, "chunks": retrieved}

# ── FAILURE 3: Wrong K — retrieving too few chunks ────────────────────────
@traceable(name="failure-3-k-equals-1")
def failure_k_too_small(question: str) -> dict:
    """
    Bug: k=1 — only one chunk retrieved.
    Expected failure: misses relevant context spread across multiple chunks.
    """
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks   = splitter.split_text(full_text)
    model, index = build_index(chunks)
    retrieved, distances = retrieve(question, model, index, chunks, k=1)
    context  = "\n\n---\n\n".join(retrieved)
    answer   = generate(question, context,
                        "Answer based only on this context:\n{context}")
    return {"answer": answer, "chunks": retrieved}

# ── CORRECT baseline ───────────────────────────────────────────────────────
@traceable(name="baseline-correct")
def baseline(question: str) -> dict:
    """Correct configuration — chunk_size=500, k=3, grounded prompt."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks   = splitter.split_text(full_text)
    model, index = build_index(chunks)
    retrieved, distances = retrieve(question, model, index, chunks, k=3)
    context  = "\n\n---\n\n".join(retrieved)
    answer   = generate(question, context,
                        "Answer based strictly on this context. "
                        "If not in context say so.\n\nContext:\n{context}")
    return {"answer": answer, "chunks": retrieved}

# ── Run all four ───────────────────────────────────────────────────────────
question = "What are the principles of good data architecture?"

print(f"Question: {question}\n")
print("Running 4 configurations — check LangSmith for traces\n")

configs = [
    ("Tiny chunks (size=50)",   failure_tiny_chunks),
    ("Vague prompt",            failure_vague_prompt),
    ("K=1 (too few chunks)",    failure_k_too_small),
    ("Baseline (correct)",      baseline),
]

for name, func in configs:
    print(f"── {name} ──")
    result = func(question)
    print(f"Answer: {result['answer'][:200]}...")
    print()

Question: What are the principles of good data architecture?

Running 4 configurations — check LangSmith for traces

── Tiny chunks (size=50) ──


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7881.15it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Answer: The principles of good data architecture are designed to ensure that an organization's data is well-organized, scalable, and easily accessible. Here are some key principles:

1. **Data Governance**: E...

── Vague prompt ──


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6987.11it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Answer: The principles of good data architecture are essential to ensure that your data management system is scalable, efficient, and maintainable. Here are some key principles:

1. **Data Integration**: The ...

── K=1 (too few chunks) ──


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6554.30it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Answer: The text only provides information on one principle of good data architecture, which is:

Principle 1: Choose Common Components Wisely...

── Baseline (correct) ──


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5994.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Answer: Unfortunately, the provided text doesn't explicitly list all the principles of good data architecture. However, it does mention that there are several key elements of "good" data architecture:

1. Ser...



In [8]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')

from langsmith import Client
from langsmith import traceable
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import sys
sys.path.append('..')
from src.rag import RAGPipeline

# ── Initialize LangSmith client ────────────────────────────────────────────
client = Client()

# ── Define three prompt versions ──────────────────────────────────────────
# Each version is an experiment — did this prompt change improve things?

PROMPT_V1 = """Answer the question based on the context below.

Context:
{context}"""

PROMPT_V2 = """You are a helpful assistant. Answer based strictly on the 
provided context. If the answer is not in the context, say exactly:
'I don't find this in the document.'

Context:
{context}"""

PROMPT_V3 = """You are DocMind, an expert document analyst. Your job is to 
answer questions accurately using ONLY the provided document context.

Rules:
1. Only use information explicitly stated in the context
2. Quote relevant passages when helpful  
3. If the answer spans multiple context sections, synthesize them
4. If not found, say: 'This information is not in the document'
5. Be concise — answer in 2-3 sentences maximum

Context:
{context}"""

# ── Load RAG pipeline ──────────────────────────────────────────────────────
rag = RAGPipeline()
rag.load_pdf("../data/Fundamentals of Data Engineering.pdf")

def run_with_prompt(question: str, prompt_template: str, version: str) -> dict:
    """Run a query with a specific prompt version — traced with version tag."""
    llm    = ChatOllama(model="llama3.2")
    parser = StrOutputParser()

    # Get retrieved chunks
    import numpy as np
    query_emb          = rag.embedding_model.encode([question]).astype(np.float32)
    distances, indices = rag.index.search(query_emb, 3)
    chunks             = [rag.chunks[i] for i in indices[0]]
    context            = "\n\n---\n\n".join(chunks)

    # Build chain with this prompt version
    prompt = ChatPromptTemplate.from_messages([
        ("system", prompt_template),
        ("human",  "{question}")
    ])
    chain  = prompt | llm | parser

    # Invoke with version metadata in run name
    answer = chain.invoke(
        {"context": context, "question": question},
        config={"run_name": f"prompt-{version}",
                "tags":     [version, "prompt-experiment"]}
    )
    return {"version": version, "answer": answer, "chunks": chunks}

# ── Test same question across all three prompt versions ────────────────────
questions = [
    "What are the principles of good data architecture?",
    "What is the difference between batch and streaming data?",
]

print("Running prompt version experiment...")
print("Same questions, three prompt versions — compare in LangSmith\n")

versions = [
    ("v1-minimal",  PROMPT_V1),
    ("v2-grounded", PROMPT_V2),
    ("v3-detailed", PROMPT_V3),
]

for question in questions:
    print(f"Question: {question}")
    print("─" * 60)
    for version, prompt in versions:
        result = run_with_prompt(question, prompt, version)
        print(f"\n{version}:")
        print(f"{result['answer'][:200]}...")
    print()

print("Go to LangSmith → filter by tag 'prompt-experiment'")
print("Compare v1, v2, v3 answers for the same question")
print("Which prompt produces the most grounded, useful answers?")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9379.56it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running prompt version experiment...
Same questions, three prompt versions — compare in LangSmith

Question: What are the principles of good data architecture?
────────────────────────────────────────────────────────────

v1-minimal:
The principles of good data architecture include:

1. Serving business requirements with a common, widely reusable set of building blocks
2. Maintaining flexibility
3. Making appropriate trade-offs
4....

v2-grounded:
The principles of good data architecture are not explicitly listed in the provided text. However, Principle 1: Choose Common Components Wisely (Page 78) is mentioned.

Although it's not a list of prin...

v3-detailed:
The document defines "good" data architecture as serving business requirements with a common, widely reusable set of building blocks while maintaining flexibility and making appropriate trade-offs. It...

Question: What is the difference between batch and streaming data?
──────────────────────────────────────────────────────────